# AI/ML Medium-Range Forecast Bust Detection & Error Reduction
### SIH_1 - Weather Forecast Uncertainty & AI Risk-Assessment Layer

This notebook explores:
1. NWP Medium-Range Forecast (Day 1 - Day 10) Error Dynamics
2. Rapidly Evolving Weather Regimes (Monsoon Depressions, Heavy Rainfall, Cyclones, Heat Waves)
3. Operational Forecast Bust Thresholds & Climatological Error Statistics
4. Dual XGBoost ML Models (Bust Probability Classifier & Error Regressor)
5. SHAP Explainability & Top Meteorological Attribution Drivers
6. **Before vs After Error Reduction Quantification**

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root in sys.path
sys.path.append(os.path.abspath('..'))
from preprocessing.read_nwp import generate_synthetic_meteorological_dataset, INDIAN_REGIONS
from preprocessing.align_data import align_forecast_and_observations
from preprocessing.calculate_error import calculate_errors_and_busts, generate_historical_error_climatology
from features.feature_engineering import extract_features, METEOROLOGICAL_FEATURE_COLS

print('Modules loaded successfully!')

## 1. Generate & Align Historical NWP & Ground-Truth Observations

In [ ]:
fc_df, obs_df = generate_synthetic_meteorological_dataset(n_samples=10000, random_seed=42)
aligned_df = align_forecast_and_observations(fc_df, obs_df)
error_df = calculate_errors_and_busts(aligned_df)
print('Aligned samples:', len(error_df))
error_df[['valid_date', 'region_id', 'lead_time_days', 'rain_forecast', 'rain_observed', 'rain_error', 'is_forecast_bust']].head(5)

## 2. Lead Time Error Growth & Bust Rate (Day 1 to Day 10)

In [ ]:
lead_stats = error_df.groupby('lead_time_days').agg(
    rain_mae=('rain_abs_error', 'mean'),
    temp_mae=('temp_abs_error', 'mean'),
    bust_rate=('is_forecast_bust', 'mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(lead_stats['lead_time_days'], lead_stats['rain_mae'], 'b-o', label='Rainfall MAE (mm/day)')
ax1.plot(lead_stats['lead_time_days'], lead_stats['temp_mae'], 'g-s', label='Temperature MAE (°C)')
ax2.plot(lead_stats['lead_time_days'], lead_stats['bust_rate']*100, 'r--^', label='Bust Frequency (%)')
ax1.set_xlabel('Lead Time (Days)')
ax1.set_ylabel('Forecast Error (MAE)')
ax2.set_ylabel('Forecast Bust Rate (%)')
plt.title('NWP Forecast Degradation & Bust Vulnerability over Day 1 - Day 10')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Train Dual XGBoost Models & AI Error Correction

In [ ]:
from training.train import train_bust_system
bundle, summary = train_bust_system(n_samples=10000)
print('Training complete! Key metrics summary:')
import json
print(json.dumps(summary['error_reduction'], indent=2))

## 4. SHAP Explainability for High-Risk Forecasts

In [ ]:
from explainability.shap_analysis import MeteoSHAPExplainer
explainer = MeteoSHAPExplainer(bundle)
sample_case = pd.DataFrame([{
    'rain_forecast': 140.0,
    'temp_forecast': 27.5,
    'mslp_forecast': 991.0,
    'rh_forecast': 96.0,
    'wind_forecast': 22.0,
    'cape_forecast': 3200.0,
    'lead_time_days': 6,
    'ensemble_spread': 16.5,
    'regime_monsoon_depression': 1,
    'is_rapid_evolving_system': 1,
    'mslp_tendency_24h': -7.5
}])
exp = explainer.explain_instance(sample_case)
print('Meteorological Drivers:')
for line in exp['meteorological_narratives']:
    print('-', line)